### Model Experiments

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from src.visualization import silhouette_plot
from src.topic_analysis import *
import pickle

In [0]:
df = spark.table("scientific_trend_analysis.core.arxiv_cleaned_sampled")
df_label = df.toPandas()

In [0]:
processed_text_list = [row['processed_text'] for row in df.select('processed_text').collect()]
print(f"Loaded {len(processed_text_list)} processed texts")

In [0]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=5, max_features=30000, sublinear_tf=True, stop_words='english', ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(processed_text_list)
feature_name = tfidf_vectorizer.get_feature_names_out()
print(feature_name)

selector = VarianceThreshold(threshold=0.00005)
X_tfidf_reduced = selector.fit_transform(X_tfidf)
feature_name_reduced = feature_name[selector.get_support()]
print(feature_name_reduced)

print("Shape before tfidf: ", X_tfidf.shape)
print("Shape after tfidf and variance threshold:", X_tfidf_reduced.shape)

In [0]:
n_components = 100
svd = TruncatedSVD(n_components=n_components, random_state=2)

lsa_result = svd.fit_transform(X_tfidf_reduced)
lsa_result_nor = normalize(lsa_result, norm='l2', axis=1)
print(f"LSA result shape: {lsa_result_nor.shape}")

explained_variance = svd.explained_variance_ratio_.sum()
print(f"Total explained variance: {explained_variance * 100:.2f}%")

In [0]:
# Save LSA features for visualization (numpy array)
dbutils.fs.mkdirs("/Volumes/scientific_trend_analysis/core/model_artifacts/")
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_features.pkl", "wb") as f:
    pickle.dump(lsa_result, f)
dbutils.fs.mkdirs("/Volumes/scientific_trend_analysis/core/model_artifacts/")
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_features_nor.pkl", "wb") as f:
    pickle.dump(lsa_result_nor, f)

#### K-Means

In [0]:
inertia = []
silhouette_scores = {}

k_range = range(5, 13)
for i in k_range:
    km = KMeans(n_clusters=i, 
                init='k-means++',
                n_init='auto',
                random_state=2,
                max_iter=300)
    km.fit(lsa_result_nor)
    inertia.append(km.inertia_)
    cluster_labels = km.predict(lsa_result_nor)
    
    score = silhouette_score(lsa_result_nor, cluster_labels)
    silhouette_scores[i] = score
    print(f"k={i}, Silhouette Score: {score:.4f}")

plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of cluster', fontsize=14)
plt.ylabel('Inertia', fontsize=14)
plt.xticks(list(k_range))

# find the best k
best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k values according to Silhouette Score: {best_k}")


In [0]:
# final km model
final_kmeans = KMeans(n_clusters=best_k, random_state=2, init='k-means++', n_init='auto', max_iter=300)
cluster_labels_km = final_kmeans.fit_predict(lsa_result_nor)

# Assign labels to the original dataframe
df_label['Cluster_ID_km'] = cluster_labels_km

silhouette_plot(lsa_result=lsa_result_nor, cluster_labels=cluster_labels_km, best_k=best_k)

In [0]:
print("K-means")
print("-" * 100)
get_top_frequency_words_per_topic(df_label, 'Cluster_ID_km')
get_top_tfidf_words_per_topic(df_label, X_tfidf_reduced, feature_name_reduced, 'Cluster_ID_km')

#### Gaussian Mixture Model

In [0]:

k_range = range(5, 13)

aic_scores, bic_scores = [], []
silhouette_scores = {}
for k in k_range:
    gmm_model = GaussianMixture(
        n_components=k,
        covariance_type='diag',
        random_state=2,
        max_iter=100,
        n_init=10
    )
    gmm_model.fit(lsa_result)
    aic_scores.append(gmm_model.aic(lsa_result))
    bic_scores.append(gmm_model.bic(lsa_result))

    gmm_cluster_labels = gmm_model.predict(lsa_result)

    score = silhouette_score(lsa_result, gmm_cluster_labels)
    silhouette_scores[k] = score 

    print(f"GMM K={k}, AIC: {gmm_model.aic(lsa_result):.2f}, BIC: {gmm_model.bic(lsa_result):.2f}, Silhouette Score: {score:.4f}")


# find the best k
best_k_gmm = max(silhouette_scores, key=silhouette_scores.get)
best_k_bic = k_range[np.argmin(bic_scores)]
print(f"\nBest k values according to Silhouette Score: {best_k_gmm}")
print(f"\nBest k values according to BIC: {best_k_bic}")

In [0]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot AIC and BIC scores on the first y-axis
ax1.plot(k_range, aic_scores, marker='o', label='AIC')
ax1.plot(k_range, bic_scores, marker='o', label='BIC')

# Add silhouette score to the plot on a second y-axis
silhouette_score_values = [silhouette_scores[k] for k in k_range]
ax2 = ax1.twinx()
ax2.plot(k_range, silhouette_score_values, marker='o', color='green', label='Silhouette Score')

best_k_bic = k_range[np.argmin(bic_scores)]
ax1.axvline(x=best_k_bic, color='red', linestyle='--', label=f'Best K (BIC): {best_k_bic}')

ax1.set_xlabel('Num. of cluster (K)', fontsize=14)
ax1.set_ylabel('AIC / BIC', fontsize=14)
ax2.set_ylabel('Silhouette Score', fontsize=14)
ax1.set_title('GMM Model Selection Metrics', fontsize=16)

# Combine legends from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='best')

plt.show()

In [0]:
# final gmm model
optimal_k = best_k_bic
final_gmm_model = GaussianMixture(
    n_components=optimal_k,
    covariance_type='diag',
    random_state=2,
    max_iter=500,
    n_init=10
)
cluster_labels_gmm = final_gmm_model.fit_predict(lsa_result)
probabilities = final_gmm_model.predict_proba(lsa_result)

# Assign labels to the original dataframe
df_label['Cluster_ID_gmm'] = cluster_labels_gmm

silhouette_plot(lsa_result=lsa_result, cluster_labels=cluster_labels_gmm, best_k=optimal_k)

In [0]:
plt.figure(figsize=(12, 6))
sns.heatmap(probabilities, cmap='viridis', cbar=True)
plt.xlabel('Cluster')
plt.ylabel('Document')
plt.title('Document-Cluster Probability Heatmap (GMM)')
plt.show()

In [0]:
print("Gaussian Mixture Model")
print("-" * 100)
get_top_frequency_words_per_topic(df_label, 'Cluster_ID_gmm')
get_top_tfidf_words_per_topic(df_label, X_tfidf_reduced, feature_name_reduced, 'Cluster_ID_gmm')

In [0]:
display(df_label)

In [0]:
spark.createDataFrame(df_label).write.mode("overwrite").saveAsTable("scientific_trend_analysis.rep.arxiv_clustered")
print(f"Saved {len(df_label)} records with cluster assignments")